# Notebook 05 — Conclusion & Cross-Task Synthesis

This notebook closes the project. The previous four notebooks each tackled one piece in isolation:

| Notebook | Task | Question |
|----------|------|----------|
| 01 | Data engineering | Merge salaries, advanced stats, and career info into one clean panel of player-seasons. |
| 02 | **Binary classification** | Can box-score stats sort players into on-court *production* tiers (BPM cutoffs)? |
| 03 | **Multi-class classification** | Can the same stats predict a player's *pay* tier (salary percentile)? |
| 04 | **Clustering** | Do the five traditional positions emerge naturally from the stats? |

Here we (1) recap the headline results of each task, (2) run the **cross-task analysis** promised in Notebook 03. We put *production* (Task 1) and *pay* (Task 2) side by side on the same held-out players to ask where the market and the box score agree and disagree. Lastly, we show the fold in the positionless finding from Task 3, then state the project's overall conclusions.


## 1. Recap of the Three Tasks

The numbers below are the headline test-set results reported at the end of Notebooks 02–04. They are reproduced here for reference; see each notebook for the full per-model, per-cutoff breakdowns.

### Task 1 — Binary classification at four BPM cutoffs (Notebook 02)
Three classifiers (Logistic Regression, Random Forest, Gradient Boosting), each tuned with 5-fold CV, evaluated at four production thresholds. **Every one of the twelve model–cutoff combinations cleared macro F1 ≥ 0.78 and AUC ≥ 0.89.** The best model shifts with cutoff difficulty: Logistic Regression wins the easier *Rotation* (F1 = 0.827) and *Starter* (F1 = 0.854) cutoffs, while Gradient Boosting wins the harder *Star* (F1 = 0.850) and *Superstar* (F1 = 0.880) cutoffs — i.e. non-linear models pull ahead as the positive class shrinks and the decision boundary curves.

### Task 2 — Multi-class salary-tier classification (Notebook 03)
The same feature matrix and train/test split, now predicting one of five salary tiers (within-season percentile). Gradient Boosting led (63.1% accuracy, macro F1 0.598, macro AUC 0.892), narrowly over Random Forest (62.4% / 0.585 / 0.891), with Logistic Regression well behind (54.5% / 0.498 / 0.843). The **9-point gap** between linear and tree models — much larger than in Task 1 — says pay depends on *interactions* of skills and demographics, not just additive production volume.

### Task 3 — Unsupervised clustering vs. position (Notebook 04)
K-means on the position-free feature matrix. Silhouette peaked at k = 2 and fell monotonically; Adjusted Rand Index against the five positions peaked only weakly at k = 5 and stayed low everywhere. **No k recovers the traditional positions** — the statistical structure of the modern game is not five discrete positions.


## 2. What is a player worth? Predicting market value from production

The three tasks above each used **all** the features, including `Age`. To find genuinely *new* information we change the question: train a model to predict **salary from production alone** — points, rebounds, assists, shooting, minutes — with **`Age` and `experience` removed** so it cannot lean on career stage. Forced to price players on production, the model learns the league's **production → salary mapping**: *given what you do on the court, what does the NBA typically pay a player like you?*

That prediction is a player's **production-based market value.** Comparing it to **actual salary** is the finding — a large gap means a player is paid far more (or far less) than their on-court production commands. We build two views:

- **Graph 1** — *expected production* vs *actual salary*: how good we expect you to be, against what you're paid.
- **Graph 2** — *actual salary* vs *predicted salary*: your real paycheck against your production-based market value, with distance from the diagonal as the over/under-payment.

Both models are production-only and produced **out-of-fold** (no player's own data leaks into their prediction). All salaries are expressed as **% of that season's salary cap** to remove cap inflation across 2010–2025.

In [11]:
import pandas as pd, numpy as np
from sklearn.model_selection import cross_val_predict, StratifiedKFold, KFold
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import r2_score

df = pd.read_csv('data/nba_clean.csv')
idc = ['Player','player_id','Year','Team']
lbl = ['bpm','bpm_rotation','bpm_starter','bpm_star','bpm_superstar','Salary','salary_tier']
num = [c for c in df.columns if c not in idc + lbl + ['Pos_primary']]
pos = pd.get_dummies(df['Pos_primary'], prefix='pos', drop_first=True).astype(int)

# PRODUCTION-ONLY feature matrix: drop Age + experience (career stage)
X = pd.concat([df[num], pos], axis=1).fillna(0).drop(columns=['Age','experience'])

CAP = {2010:57.7,2011:58.044,2012:58.044,2013:58.044,2014:58.679,2015:63.065,2016:70.0,2017:94.143,
       2018:99.093,2019:101.869,2020:109.14,2021:109.14,2022:112.414,2023:123.655,2024:136.021,2025:140.588}
df['cap_pct'] = df['Salary'] / (df['Year'].map(CAP) * 1e6)        # actual salary, % of cap

cv = StratifiedKFold(5, shuffle=True, random_state=42)
kf = KFold(5, shuffle=True, random_state=42)

# Expected production tier (0–4) from production, out-of-fold
df['exp_prod'] = np.zeros(len(df))
for c in ['bpm_rotation','bpm_starter','bpm_star','bpm_superstar']:
    df['exp_prod'] += cross_val_predict(HistGradientBoostingClassifier(random_state=42),
                                        X.values, df[c].values, cv=cv, method='predict_proba')[:, 1]

# Predicted salary (% of cap) from production, out-of-fold regression
df['pred_cap'] = cross_val_predict(HistGradientBoostingRegressor(random_state=42),
                                   X.values, df['cap_pct'].values, cv=kf)
df['resid'] = df['cap_pct'] - df['pred_cap']     # >0 overpaid, <0 underpaid

r2 = r2_score(df['cap_pct'], df['pred_cap'])
print(f"Salary-from-production model R^2 (out-of-fold): {r2:.3f}")
print(f"Production explains ~{r2:.0%} of salary; the leftover (residual) is the over/under-payment.")

Salary-from-production model R^2 (out-of-fold): 0.549
Production explains ~55% of salary; the leftover (residual) is the over/under-payment.


### 2.1 Expected production vs. actual salary

**x = expected production tier** (how good the box score says you are), **y = actual salary** (% of cap), colored by over/under-payment (red = overpaid, blue = underpaid). A fair market would rise along a diagonal. The breaks are the **upper-left** (low production, high pay → overpaid) and **lower-right** (star production, tiny pay → underpaid).

In [14]:
import plotly.express as px

def market_scatter(xcol, xlabel, title, diagonal=False):
    fig = px.scatter(df, x=xcol, y='cap_pct', color='resid',
        color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
        custom_data=['Player','Team','Year','exp_prod','bpm','PTS','Salary','cap_pct','pred_cap','resid'],
        opacity=0.7, labels={xcol:xlabel, 'cap_pct':'actual salary (% of cap)', 'resid':'over/under'},
        title=title)
    fig.update_traces(marker=dict(size=6, line=dict(width=0.3, color='white')),
        hovertemplate=('<b>%{customdata[0]}</b> — %{customdata[1]}, %{customdata[2]}<br>'
            'expected production: %{customdata[3]:.2f}/4  (BPM %{customdata[4]:.1f}, PTS %{customdata[5]:.1f})<br>'
            'actual salary: $%{customdata[6]:,.0f} = %{customdata[7]:.1%} of cap<br>'
            'production-based market value: %{customdata[8]:.1%} of cap<br>'
            'over/under: %{customdata[9]:+.1%}<extra></extra>'))
    fig.update_layout(width=950, height=620, template='plotly_white',
        yaxis=dict(tickformat='.0%'),
        coloraxis_colorbar=dict(title='over / under<br>(actual − value)', tickformat='+.0%'))
    if diagonal:
        hi = max(df[xcol].max(), df['cap_pct'].max())
        fig.add_shape(type='line', x0=0, y0=0, x1=hi, y1=hi, line=dict(dash='dash', color='gray'))
        fig.update_xaxes(tickformat='.0%')
        fig.add_annotation(x=0, y=hi, xanchor='left', yanchor='top',
                           text='OVERPAID<br>(bad deals)', showarrow=False, font=dict(color='#a50f15', size=11))
        fig.add_annotation(x=hi, y=0, xanchor='right', yanchor='bottom',
                           text='UNDERPAID', showarrow=False, font=dict(color='#08519c', size=11))
    return fig

market_scatter('exp_prod', 'expected production tier (0–4)',
               'Graph 1 — expected production vs. actual salary').show()

The cloud rises — better players are paid more — but the spread is enormous. **Upper-left (red):** high pay, low expected production → overpaid. **Lower-right (blue):** star production for a sliver of the cap → underpaid (nearly all rookie-scale deals). This shows the *relationship*; to read off a dollar verdict we put salary on **both** axes.

### 2.2 Actual vs. predicted salary — who is overpaid and underpaid

Now **x = predicted salary** (production-based market value) and **y = actual salary**, both % of cap, with the **dashed diagonal = fairly paid.** Vertical distance from the line is the verdict — and it confirms the geometry exactly: the worst deals sit on the **left** (low production-based value) and **high up** (paid a lot for it), i.e. **upper-left, above the line.** Underpaid players fall lower-right.

In [15]:
market_scatter('pred_cap', 'predicted salary — production-based market value: (% of cap)',
               'Graph 2 — actual vs. predicted (production-based) salary', diagonal=True).show()

In [4]:
cols = ['Player','Year','Team','exp_prod','bpm','PTS','cap_pct','pred_cap','resid','Salary']
fm = {'exp_prod':'{:.2f}','bpm':'{:.1f}','PTS':'{:.1f}','cap_pct':'{:.1%}',
      'pred_cap':'{:.1%}','resid':'{:+.1%}','Salary':'${:,.0f}'}
f = {k: v.format for k, v in fm.items()}
print('MOST OVERPAID — actual salary far ABOVE production-based value (the bad deals):')
print(df.sort_values('resid', ascending=False).head(12)[cols].to_string(index=False, formatters=f))
print('\nMOST UNDERPAID — production-based value far above actual salary:')
print(df.sort_values('resid').head(12)[cols].to_string(index=False, formatters=f))

MOST OVERPAID — actual salary far ABOVE production-based value (the bad deals):
           Player  Year Team exp_prod  bpm  PTS cap_pct pred_cap  resid      Salary
        John Wall  2023  LAC     0.76 -1.2 11.4   40.6%     6.4% +34.2% $50,200,141
Russell Westbrook  2023  LAC     1.01  0.2 15.9   40.4%     7.8% +32.6% $49,918,548
    Tracy McGrady  2010  NYK     0.11 -2.2  8.2   39.6%     8.0% +31.6% $22,843,124
Amar'e Stoudemire  2014  NYK     0.93 -1.6 11.9   36.9%     5.7% +31.2% $21,679,893
Russell Westbrook  2022  LAL     1.01 -1.6 18.5   45.5%    14.3% +31.2% $51,123,449
      Kobe Bryant  2013  LAL     3.30  4.6 27.3   52.5%    23.2% +29.2% $30,453,805
    Blake Griffin  2022  BRK     0.82 -0.5  6.4   33.3%     4.4% +28.9% $37,472,385
  Jermaine O'Neal  2010  MIA     0.99 -0.1 13.6   39.9%    11.7% +28.2% $23,016,000
     Kemba Walker  2022  NYK     0.92  0.4 11.6   36.0%     7.9% +28.1% $40,434,514
        John Wall  2021  HOU     1.23 -0.7 20.6   46.1%    20.5% +25.6% $50,277,

### 2.3 The finding

The residual - **actual salary minus production-based market value** - is a concrete, model-derived fair-value stat, and its extremes make logical sense:

- **Overpaid:** **John Wall (2023 — paid 41% of the cap, production worth ~6%), Russell Westbrook (2022–23), Tracy McGrady, Amar'e Stoudemire, Blake Griffin, Kemba Walker** — almost all **injured or sharply declining veterans on max contracts signed years earlier.** Their current box score no longer justifies the salary, and the model flags it purely from the weak production.
- **Underpaid:** **Tyrese Haliburton, Desmond Bane, Giannis (2017), Jokić (2018), Shai Gilgeous-Alexander, Zion, Anthony Davis** — **young stars on rookie-scale deals** producing like franchise players for ~3–5% of the cap while their production commands ~30%.

This is the payoff. A model that learned the league's production→salary mapping, with career stage stripped out, returns each player's **fair market value from production alone** and measures the gap to their real paycheck — cleanly separating genuinely bad contracts from the structural bargains of the rookie scale. It is a model-derived valuation, **not** arithmetic on the salary column.

## 3. Folding in Task 3: Why Position Barely Mattered

Task 3 found the five positions don't emerge from the stats. That isn't an isolated curiosity — it explains a pattern in Tasks 1 and 2: the one-hot **position features carried little predictive weight**. If positions were statistically distinct clusters, they'd be strong predictors of both production and pay. They aren't, because the modern game is positionless — a stretch four and a small-ball five occupy overlapping statistical space.

We confirm it directly: how much does position alone separate BPM and salary, compared to the spread *within* each position?


In [5]:
by_pos = df.groupby('Pos_primary').agg(
    n=('bpm', 'size'),
    bpm_mean=('bpm', 'mean'),
    bpm_std=('bpm', 'std'),
    salary_med=('Salary', 'median'),
).round(2)
print(by_pos)

# Between-position vs within-position variance in BPM (eta^2-style)
grand = df['bpm'].mean()
ss_between = df.groupby('Pos_primary')['bpm'].apply(lambda g: len(g) * (g.mean() - grand) ** 2).sum()
ss_total = ((df['bpm'] - grand) ** 2).sum()
print(f"\nShare of BPM variance explained by position alone: {ss_between / ss_total:.1%}")
print("-> Position explains very little; production lives WITHIN positions, not BETWEEN them.")

                n  bpm_mean  bpm_std  salary_med
Pos_primary                                     
C            1169     -0.09     2.80   4350000.0
PF           1203     -0.62     2.68   4500000.0
PG           1164     -0.27     3.10   3920066.0
SF           1126     -0.97     2.74   3310614.5
SG           1358     -1.12     2.43   3529887.5

Share of BPM variance explained by position alone: 2.0%
-> Position explains very little; production lives WITHIN positions, not BETWEEN them.


Position accounts for only a sliver of the variance in production. The overwhelming majority of what separates players lives *within* each position label — which is exactly why box-score features (Tasks 1 & 2) classify well while position labels (Task 3) don't form clean clusters. The three tasks are three views of the same underlying fact: **what a player does matters far more than what position they're listed at.**


## 4. Overall Conclusions

1. **Production is highly learnable from the box score.** Every model–cutoff combination in Task 1 cleared macro F1 ≥ 0.78 / AUC ≥ 0.89. On-court value, as measured by BPM, is almost fully recoverable from per-game statistics.

2. **Pay is learnable but harder, and non-linear.** Salary-tier prediction tops out around 63% (five-class, vs. 20% chance), reflecting that compensation depends on interactions of skills, role, and career stage — not raw production alone.

3. **A production-only salary model exposes real over/under-payment — the project's new result.** Training a model to predict salary from production alone (age and experience removed, out-of-fold R² ≈ 0.55) yields each player's *production-based market value*. The gap to actual salary flags the league's worst contracts — injured/declining veterans on legacy max deals (John Wall, Westbrook, McGrady, Amar'e) — and its biggest bargains — rookie-scale stars (Haliburton, Giannis, Jokić, SGA) producing like franchise players for a fraction of the cap. This residual is a model-derived 'fair-value' stat, not arithmetic on the salary column.

4. **The five traditional positions are not in the data.** Clustering recovers no positional structure, and position explains only a sliver of production variance. The modern NBA is positionless, and our supervised models succeed *because* they lean on what players do rather than what they're labeled.
